###Tansform Results data
1. Read bronze results table
2. Keep only columns required for analytics (drop url column)
3. Standardise column names using snake case
4. Rename columns to make them more meaningful
5. Filter out rows where season, round, constructor_id, or driver_id is null
6. Remove duplicate records
7. Transform values of column race_name to Title Case
8. Write the Transformed data to silver results table

In [0]:
%run ../00-Common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.results"
silver_table = f"{catalog_name}.{silver_schema}.results"

In [0]:
from pyspark.sql import functions as f

In [0]:
results_df = (
    spark.read.option("versionAsOf", 0).table(bronze_table)
        .drop("url")
        .withColumnsRenamed({
        "constructorId": "constructor_id",
        "date": "race_date",
        "driverId": "driver_id",
        "grid": "grid_position",
        "laps": "laps_completed",
        "number": "car_number",
        "position": "finish_position",
        "postitonText": "position_text",
        "raceName": "race_name"
    })
)

In [0]:
results_filtered_df =(
    results_df
        .filter(
            f.col("season").isNotNull() &
            f.col("round").isNotNull() &
            f.col("constructor_id").isNotNull() &
            f.col("driver_id").isNotNull()
        )
        .dropDuplicates(["season", "round", "driver_id", "constructor_id"])
)

In [0]:
display(results_df.count() - results_filtered_df.count())

In [0]:
results_final_df = (
    results_filtered_df
        .withColumn("race_name", f.initcap(f.col("race_name")))
)

In [0]:
(
    results_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))